### Clone the repo and install dependencies

In [ ]:
%cd /content
!git clone https://github.com/Comfy-Org/ComfyUI.git
%cd /content/ComfyUI
!pip install -r requirements.txt
!git clone https://github.com/Comfy-Org/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
!pip install -r custom_nodes/ComfyUI-Manager/requirements.txt


### Connect Google Drive to store downloaded models and load them

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')

for folder in ('checkpoints', 'loras', 'vae', 'diffusion_models', 'text_encoders', 'clip'):
    (MODELS_DIR / folder).mkdir(parents=True, exist_ok=True)

# Add the Drive folders to the ComfyUI config without removing other settings.
config = Path('/content/ComfyUI/extra_model_paths.yaml')
existing = config.read_text() if config.exists() else ''

if 'drive_models:' not in existing:
    drive_config = """drive_models:
  base_path: /content/drive/MyDrive/ComfyUI/models
  checkpoints: checkpoints
  loras: loras
  vae: vae
  diffusion_models: diffusion_models
  text_encoders: text_encoders
  clip: clip
"""
    config.write_text(existing.rstrip() + ('\n' if existing.strip() else '') + drive_config)

print('Drive ready:', MODELS_DIR)

### Download VAE, text encoder and diffusion model to Drive

In [ ]:
# Download the MiniMax H3 VAE, text encoder, and diffusion model to Google Drive (about 37 GB).
from pathlib import Path
import requests
from tqdm.auto import tqdm

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')
ASSETS = [

    ('vae', 'minimax_h3_audio_vae_fp32.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors'),
    ('vae', 'minimax_h3_video_vae_int8_convrot.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors'),
    ('text_encoders', 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
    ('diffusion_models', 'minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
]

session = requests.Session()
prepared = []
print('Checking links and existing files...')
for folder, filename, url in ASSETS:
    destination = MODELS_DIR / folder / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    response = session.head(url, allow_redirects=True, timeout=60)
    response.raise_for_status()
    expected = int(response.headers.get('Content-Length', 0))
    if destination.exists() and expected and destination.stat().st_size == expected:
        print(f'Already complete, skipping: {filename}')
        continue
    if destination.exists() and not expected and destination.stat().st_size > 0:
        print(f'File already exists (server size unavailable), skipping: {filename}')
        continue
    prepared.append((destination, url, expected, filename))

if not prepared:
    print('All assets are already available on Drive.')
else:
    total = sum(size for _, _, size, _ in prepared)
    print(f'{len(prepared)} file(s) to download; detected total size: {total / (1024**3):.2f} GiB.')
    for destination, url, expected, filename in prepared:
        partial = destination.with_name(destination.name + '.part')
        partial.unlink(missing_ok=True)
        try:
            with session.get(url, stream=True, allow_redirects=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with partial.open('wb') as output, tqdm(
                    total=expected or None, unit='B', unit_scale=True, unit_divisor=1024, desc=filename
                ) as progress:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            output.write(chunk)
                            progress.update(len(chunk))
            downloaded = partial.stat().st_size
            if expected and downloaded != expected:
                raise IOError(f'Size mismatch for {filename}: got {downloaded} of {expected} bytes.')
            partial.replace(destination)
            print(f'Done: {destination}')
        except Exception:
            partial.unlink(missing_ok=True)
            raise

print('Done. If ComfyUI is already running, refresh the model list or restart the server so the models are detected.')

### Download a checkpoint or LoRA to Google Drive

In [ ]:
# Fill in a Civitai or Hugging Face URL, then run this cell.
#
# DISCLAIMER (Civitai downloads):
# Set up a Civitai API key as a Colab secret named 'CIVITAI_API_TOKEN'
# (Colab left sidebar > Secrets > enable notebook access).
# It is only used so the download is not rejected with a 401 error.
URL = 'https://civitai.red/models/827184/wai-illustrious-sdxl?modelVersionId=2883731'
# For Hugging Face files in the repo root, choose the target folder here.
HF_MODEL_TYPE = 'loras'

from pathlib import Path
from urllib.parse import urlparse, parse_qs
from google.colab import userdata
import requests

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')
parsed = urlparse(URL.strip())
host = (parsed.hostname or '').lower()
if host.startswith('www.'):
    host = host[4:]
parts = parsed.path.strip('/').split('/')
query = parse_qs(parsed.query)

if host == 'huggingface.co':
    if len(parts) < 5 or parts[2] not in ('blob', 'resolve'):
        raise ValueError('Enter a Hugging Face file link in /blob/ or /resolve/ format.')
    owner, repo, _, revision = parts[:4]
    file_path = parts[4:]
    if not file_path:
        raise ValueError('The Hugging Face link does not point to a file.')
    download_url = f'https://huggingface.co/{owner}/{repo}/resolve/{revision}/' + '/'.join(file_path)
    allowed_types = {'checkpoints', 'loras', 'vae', 'diffusion_models', 'text_encoders', 'clip'}
    if file_path[0] in allowed_types:
        kind = file_path[0]
        relative_parts = file_path
    else:
        kind = HF_MODEL_TYPE.strip().lower()
        relative_parts = [file_path[-1]]
    if kind not in allowed_types:
        raise ValueError(f'HF_MODEL_TYPE must be one of: {sorted(allowed_types)}')
    destination = MODELS_DIR / kind
    for part in relative_parts:
        if part in ('', '.', '..'):
            raise ValueError('Invalid file path.')
        destination = destination / part
    headers = {}
elif host in ('civitai.com', 'civitai.red'):
    token = None
    try:
        token = userdata.get('CIVITAI_API_TOKEN')
    except Exception:
        pass
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    api_base = f'https://{host}'
    version_id = query.get('modelVersionId', [None])[0]
    file_id = query.get('fileId', [None])[0]
    is_download_link = len(parts) >= 4 and parts[-3:-1] == ['download', 'models']

    if is_download_link:
        model_id = parts[-1]
        response = requests.get(f'{api_base}/api/v1/models/{model_id}', headers=headers, timeout=30)
        response.raise_for_status()
        model_info = response.json()
        chosen = None
        for version in model_info.get('modelVersions') or []:
            for file in version.get('files') or []:
                if file_id and str(file.get('id')) == str(file_id):
                    chosen = (version, file)
                    break
            if chosen:
                break
        if chosen is None:
            version = (model_info.get('modelVersions') or [{}])[0]
            files = version.get('files') or []
            selected_file = next((f for f in files if f.get('primary')), files[0] if files else None)
            if selected_file:
                chosen = (version, selected_file)
        if not chosen:
            raise RuntimeError('Civitai did not return any model file.')
        version_info, selected_file = chosen
        download_url = URL.strip()
    else:
        if not version_id:
            raise ValueError('The Civitai link must have a modelVersionId or be a /api/download/models/ link.')
        response = requests.get(f'{api_base}/api/v1/model-versions/{version_id}', headers=headers, timeout=30)
        response.raise_for_status()
        version_info = response.json()
        model_info = version_info.get('model') or {}
        files = version_info.get('files') or []
        selected_file = next((f for f in files if file_id and str(f.get('id')) == str(file_id)), None)
        selected_file = selected_file or next((f for f in files if f.get('primary')), files[0] if files else None)
        if not selected_file:
            raise RuntimeError('Civitai did not return any file for this version.')
        model_id = model_info.get('id')
        download_url = selected_file.get('downloadUrl') or f'{api_base}/api/download/models/{model_id}?fileId={selected_file.get("id")}'

    kind_map = {'lora': 'loras', 'checkpoint': 'checkpoints', 'textualinversion': 'embeddings', 'vae': 'vae', 'controlnet': 'controlnet'}
    kind = kind_map.get((model_info.get('type') or '').lower(), 'loras')
    filename = selected_file.get('name') or 'model.safetensors'
    destination = MODELS_DIR / kind / filename
    headers['Authorization'] = f'Bearer {token}' if token else ''
else:
    raise ValueError('Unsupported host. Use civitai.com, civitai.red, or huggingface.co.')

if headers.get('Authorization') == '':
    headers.pop('Authorization')
destination.parent.mkdir(parents=True, exist_ok=True)
if destination.exists():
    print(f'Already exists, skipping: {destination} ({destination.stat().st_size:,} bytes)')
else:
    partial = destination.with_name(destination.name + '.part')
    if partial.exists():
        raise FileExistsError(f'Temporary file already exists and will not be overwritten: {partial}')
    print(f'Downloading {destination.name} to {destination}', flush=True)
    try:
        with requests.get(download_url, headers=headers, allow_redirects=True, stream=True, timeout=(60, 180)) as response:
            response.raise_for_status()
            expected = int(response.headers.get('Content-Length', 0)) or None
            downloaded = 0
            with open(partial, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
            if expected is not None and downloaded != expected:
                raise IOError(f'Download size mismatch: got {downloaded:,} of {expected:,} bytes.')
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
    print(f'Saved: {destination} ({destination.stat().st_size:,} bytes)')
print('If ComfyUI is already running, refresh the model list; restart the server only if the model does not show up.')

### Run ComfyUI with cloudflared




In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import time
import socket
import urllib.request

COMFY_DIR = Path("/content/ComfyUI")
COMFY_LOG = Path("/content/comfyui.log")
TUNNEL_LOG = Path("/content/cloudflared.log")
PORT = 8188

def port_is_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            return True
    except OSError:
        return False

# Install cloudflared only if the fresh runtime does not have it yet.
if shutil.which("cloudflared") is None:
    deb_path = "/tmp/cloudflared-linux-amd64.deb"
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        deb_path,
    )
    subprocess.run(["dpkg", "-i", deb_path], check=True)

if not COMFY_DIR.exists():
    raise FileNotFoundError(f"ComfyUI folder not found: {COMFY_DIR}")

# Keep the process as a background object so the cell finishes right away and the kernel stays free.
comfyui_process = None
if port_is_open(PORT):
    print(f"ComfyUI is already running on port {PORT}; not launching a duplicate.", flush=True)
else:
    comfy_log_file = COMFY_LOG.open("w")
    comfyui_process = subprocess.Popen(
        ["python", "main.py", "--dont-print-server"],
        cwd=COMFY_DIR,
        stdout=comfy_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    comfy_log_file.close()
    print(f"ComfyUI launched in the background (PID {comfyui_process.pid}); waiting for port {PORT}.", flush=True)

    deadline = time.time() + 180
    while time.time() < deadline and not port_is_open(PORT):
        if comfyui_process.poll() is not None:
            tail = COMFY_LOG.read_text(errors="replace")[-5000:]
            raise RuntimeError("ComfyUI stopped during startup; see log " + str(COMFY_LOG) + " | " + tail)
        time.sleep(2)
    if not port_is_open(PORT):
        raise TimeoutError(f"ComfyUI was not ready after 180 seconds. Check {COMFY_LOG}")

# Prevent duplicate tunnels if this cell is re-run in the same runtime.
active_tunnel = False
for cmdline in Path("/proc").glob("[0-9]*/cmdline"):
    try:
        args = cmdline.read_bytes().replace(b"\0", b" ").decode(errors="ignore")
        if "cloudflared" in args and "tunnel" in args and f"127.0.0.1:{PORT}" in args:
            active_tunnel = True
            break
    except OSError:
        pass

cloudflared_process = None
if active_tunnel:
    print("Cloudflare Tunnel is already running; use the URL printed earlier.", flush=True)
else:
    tunnel_log_file = TUNNEL_LOG.open("w")
    cloudflared_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=tunnel_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    tunnel_log_file.close()
    print(f"Cloudflare Tunnel launched in the background (PID {cloudflared_process.pid}).", flush=True)

    deadline = time.time() + 30
    tunnel_url = None
    while time.time() < deadline:
        if cloudflared_process.poll() is not None:
            break
        log_text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            tunnel_url = match.group(0)
            break
        time.sleep(1)
    if tunnel_url:
        print("ComfyUI URL:", tunnel_url, flush=True)
    else:
        print(f"Tunnel is still starting or has not provided a URL yet. Check log: {TUNNEL_LOG}", flush=True)

print(f"ComfyUI log: {COMFY_LOG}")
print(f"Tunnel log: {TUNNEL_LOG}")

In [ ]:
from google.colab import output
print(output.eval_js("google.colab.kernel.proxyPort(8188)"))

### Run ComfyUI with localtunnel




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

### Run ComfyUI with a Colab iframe (fallback in case localtunnel fails)

If you see a 403 error, try opening the link in another browser.

Some UI features such as live image preview may not work because the Colab iframe blocks websockets.

In [ ]:
import os, socket, subprocess, time
from google.colab import output

log = open('/tmp/comfyui.log', 'w')
server = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--dont-print-server'],
    cwd='/content/ComfyUI', stdout=log, stderr=subprocess.STDOUT, start_new_session=True
)
for _ in range(120):
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        if server.poll() is not None:
            log.flush()
            print(open('/tmp/comfyui.log').read()[-12000:])
            raise RuntimeError('ComfyUI stopped before port 8188 was ready')
        time.sleep(1)
else:
    print(open('/tmp/comfyui.log').read()[-12000:])
    raise TimeoutError('ComfyUI did not open port 8188 within 120 seconds')
output.serve_kernel_port_as_iframe(8188, height=1024)

In [ ]:
import requests
r = requests.get('http://127.0.0.1:8188/', timeout=10)
print('HTTP', r.status_code, 'bytes', len(r.content))
print(r.text[:300])

In [ ]:
import requests
urls = [
  'https://civitai.red/api/download/models/3218603?fileId=3100615',
  'https://civitai.red/api/download/models/2575595?fileId=2463002',
]
for url in urls:
    r = requests.get(url, allow_redirects=True, stream=True, timeout=45)
    print('status:', r.status_code, 'final host:', r.url.split('/')[2])
    print('type:', r.headers.get('Content-Type'), 'size:', r.headers.get('Content-Length'), 'name:', r.headers.get('Content-Disposition'))
    r.close()

In [ ]:
import os, requests
print('DRIVE_MOUNTED:', os.path.ismount('/content/drive'), 'MYDRIVE_EXISTS:', os.path.isdir('/content/drive/MyDrive'))
for version in (3218603, 2575595):
    r = requests.get(f'https://civitai.com/api/v1/model-versions/{version}', timeout=30)
    print('VERSION', version, 'HTTP', r.status_code)
    if r.ok:
        data = r.json()
        print('name:', data.get('name'), '| base:', data.get('baseModel'))
        for f in data.get('files', []):
            print('file:', f.get('name'), '| type:', f.get('type'), '| bytes:', f.get('sizeKB'), 'KB | primary:', f.get('primary'))
    else:
        print(r.text[:300])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('MOUNTED:', __import__('os').path.isdir('/content/drive/MyDrive'))

In [ ]:
from google.colab import userdata
from pathlib import Path
import requests, shutil, os, time
token=userdata.get('CIVITAI_API_TOKEN')
if not token: raise RuntimeError('Secret CIVITAI_API_TOKEN is not available')
items=[
    (3218603,3100615,'oneObsession_v24.safetensors','checkpoints',6938040682),
    (2575595,2463002,'AIイラストおじさん_V3-Exp-Compact.safetensors','loras',912618100),
]
for mid,fid,name,kind,expected in items:
    local_dir=Path('/content/ComfyUI/models')/kind
    drive_dir=Path('/content/drive/MyDrive/ComfyUI/models')/kind
    local_dir.mkdir(parents=True,exist_ok=True); drive_dir.mkdir(parents=True,exist_ok=True)
    dest=local_dir/name; drive_dest=drive_dir/name
    if dest.exists() or drive_dest.exists(): raise FileExistsError(f'Destination file already exists: {name}; not overwriting')
    part=local_dir/(name+'.part'); drive_part=drive_dir/(name+'.part')
    if part.exists() or drive_part.exists(): raise FileExistsError(f'.part file already exists: {name}; not overwriting')
    url=f'https://civitai.com/api/download/models/{mid}?fileId={fid}'
    print(f'Starting {name} ({expected:,} bytes)',flush=True)
    with requests.get(url,headers={'Authorization':f'Bearer {token}'},allow_redirects=True,stream=True,timeout=(60,180)) as r:
        if r.status_code != 200: raise RuntimeError(f'Download of {name} failed: HTTP {r.status_code}; {r.text[:200]}')
        length=r.headers.get('Content-Length')
        if length and int(length)!=expected: raise RuntimeError(f'Server size does not match for {name}: {length}')
        downloaded=0; report=0; started=time.time()
        with open(part,'wb') as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if chunk:
                    f.write(chunk); downloaded+=len(chunk)
                    if downloaded-report>=256*1024*1024:
                        print(f'  {name}: {downloaded/expected:.0%} ({downloaded:,}/{expected:,} bytes)',flush=True); report=downloaded
            f.flush(); os.fsync(f.fileno())
    if downloaded!=expected: raise RuntimeError(f'Download size mismatch for {name}: got {downloaded:,} of {expected:,} bytes')
    part.replace(dest)
    print(f'Local download finished: {dest.stat().st_size:,} bytes; copying to Drive...',flush=True)
    shutil.copyfile(dest,drive_part)
    if drive_part.stat().st_size!=expected: raise RuntimeError(f'Drive copy size mismatch for {name}')
    drive_part.replace(drive_dest)
    print(f'SAVED: {drive_dest} ({drive_dest.stat().st_size:,} bytes)',flush=True)
del token
print('DONE: both files saved to Google Drive.',flush=True)

### Last resort

In [ ]:
import socket
import subprocess
import time
from google.colab import output

def comfyui_running():
    try:
        with socket.create_connection(("127.0.0.1", 8188), timeout=2):
            return True
    except OSError:
        return False

if not comfyui_running():
    log = open("/content/comfyui.log", "a")
    process = subprocess.Popen(
        ["python", "main.py", "--dont-print-server"],
        cwd="/content/ComfyUI",
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log.close()

    for _ in range(180):
        if comfyui_running():
            break
        time.sleep(1)

if not comfyui_running():
    raise RuntimeError("ComfyUI is not running. Check the log at /content/comfyui.log")

url = output.eval_js("google.colab.kernel.proxyPort(8188)")
print("ComfyUI URL:", url)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8188, height=900)